# T2.3 – Car-Only YOLO Fine-Tuning

## 1. Setup & Imports

In [ ]:
# 1. Install YOLO
!pip install -q -U ultralytics

# Imports
import os
from google.colab import drive
from IPython.display import Image, display
from ultralytics import YOLO

drive.mount('/content/drive')


## 2. Load & Extract the Car-Only Dataset

In [ ]:
# 1. Unzip the car-only dataset
# Update the zip file name if it's different in Drive
!unzip -q "/content/drive/MyDrive/kitti3k_car_only.zip"

# 2. Flatten the folder structure (as in T2.1/T2.2)
!mv /content/kitti3k_car_only/kitti3k_car_only/* /content/kitti3k_car_only/ 2>/dev/null || true

print("Setup complete! Check the files sidebar to ensure the dataset is there.")


## 3. Train the Car-Only Model (Best T2.2 Configuration)

In [ ]:
# Path to the car-only dataset config
dataset_yaml_car = "/content/kitti3k_car_only/kitti3k_car_only.yaml"

# Load fresh pre-trained YOLO11n weights
model_t23 = YOLO("yolo11n.pt")

print("--- Starting Training: Task 2.3 (Car-Only) ---")
train_results_t23 = model_t23.train(
    data=dataset_yaml_car,
    epochs=30,          # Winning configuration from T2.2
    imgsz=640,
    batch=16,
    lr0=0.005,          # Winning configuration from T2.2
    freeze=0,            # Winning configuration from T2.2
    workers=2,
    cache="ram",
    device=0,
    exist_ok=True,
    pretrained=True,
    name="t23_car_only"
)


## 4. Evaluate on the Test Set

In [ ]:
print("--- Evaluating Car-Only Model on the TEST Set ---")

best_model_t23 = YOLO("runs/detect/t23_car_only/weights/best.pt")

test_metrics_t23 = best_model_t23.val(
    data=dataset_yaml_car,
    split="test",
    imgsz=640,
    conf=0.001,
    iou=0.7,
    exist_ok=True,
    name="t23_test_eval"
)

print("\n================ TASK 2.3 TEST SET RESULTS ================")
print(f"Car AP50:    {test_metrics_t23.box.map50:.4f}")
print(f"Car AP50-95: {test_metrics_t23.box.map:.4f}")
print("===========================================================")


## 5. Training Curves & Test Set Plots

In [ ]:
print("=== Task 2.3: Car-Only Training Curves ===")
# results.png shows the loss dropping over the 30 training epochs
train_results_path = "runs/detect/t23_car_only/results.png"
if os.path.exists(train_results_path):
    display(Image(filename=train_results_path, width=800))
else:
    print("Could not find results.png")

print("\n=== Task 2.3: Test Set Precision-Recall Curve ===")
# BoxPR_curve.png from the final test evaluation
test_pr_path = "runs/detect/t23_test_eval/BoxPR_curve.png"
if os.path.exists(test_pr_path):
    display(Image(filename=test_pr_path, width=600))
else:
    print("Could not find BoxPR_curve.png")

print("\n=== Task 2.3: Test Set Confusion Matrix ===")
# Notice how much simpler this matrix is compared to the 3-class T2.2 model
test_cm_path = "runs/detect/t23_test_eval/confusion_matrix.png"
if os.path.exists(test_cm_path):
    display(Image(filename=test_cm_path, width=600))
else:
    print("Could not find confusion_matrix.png")
